# 🚪 Сценарий отклонения: «Забыли закрыть дверь»

## Что происходит с теплицей?

Когда дверь остаётся открытой, резко возрастает **воздухообмен** с улицей:

```
НОРМАЛЬНЫЙ РЕЖИМ              ОТКРЫТАЯ ДВЕРЬ
┌─────────────┐               ┌─────────────┐
│ tAir ≈ 18°C │               │ tAir → tOut │
│ RH  ≈ 75%   │  🚪 открыта  │ RH  → ROut  │
│ CO₂ ≈ 800pp │ ────────────►│ CO₂→ 400ppm │
│ cLeakage    │               │ cLeakage    │
│   = 1×10⁻⁴  │               │   = 0.05    │
└─────────────┘               └─────────────┘
      ↑ утечка 0.0001 м³/м²/с       ↑ утечка в 500 раз больше!
```

В GreenLight воздухообмен с улицей описывается параметром **`cLeakage`** (коэф. утечки).  
Открытая дверь увеличивает его в **~500 раз**: `1×10⁻⁴ → 5×10⁻²`

Физические последствия:
- 🌡 **Температура** воздуха падает к уличной
- 💧 **Влажность** меняется до уличной (октябрь → холодно + туман)
- 🌿 **CO₂** разбавляется до атмосферного уровня (~400 ppm)
- 🍅 **Рост урожая** замедляется или останавливается

## 1. Подготовка — запускаем ОБЕ симуляции

In [ ]:
import sys, os, shutil, json
sys.path.insert(0, '/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from datetime import datetime, timedelta
import ipywidgets as widgets
from ipywidgets import interact, HBox, VBox, Layout
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

from greenlight import GreenLight

# ── Стиль графиков (тёмный) ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',   'axes.labelcolor': '#e6edf3',
    'text.color': '#e6edf3',       'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',      'grid.color': '#21262d',
    'grid.linewidth': 0.8,         'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d', 'legend.labelcolor': '#e6edf3',
    'font.size': 11,               'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})

# ── Пути ─────────────────────────────────────────────────────────────────────
PKG = '/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/greenlight/models'
MAIN_JSON  = PKG + '/katzin_2021/definition/main_katzin_2021.json'
DATA_CSV   = PKG + '/katzin_2021/input_data/test_data/Bleiswijk_from_20091020.csv'
MOD_DIR    = '/tmp/gl_open_door'
EXT_FILE   = MOD_DIR + '/katzin_2021/definition/extension_greenhouse_katzin_2021_vanthoor_2011.json'
MOD_MAIN   = MOD_DIR + '/katzin_2021/definition/main_katzin_2021.json'

T0 = datetime(2009, 10, 20)  # начало данных Bleiswijk

def run_sim(main_json, data_csv, t_days=1, rtol='1e-3', atol='1e-1'):
    """Запустить симуляцию и вернуть DataFrame с результатами"""
    mdl = GreenLight(input_prompt=[main_json, data_csv])
    mdl.options['t_end']       = str(int(t_days * 24 * 3600))
    mdl.options['output_step'] = '300'   # каждые 5 минут
    mdl.options['rtol']        = rtol
    mdl.options['atol']        = atol
    mdl.load()
    mdl.solve()
    df = mdl.full_sol.copy()
    df['hours'] = df['Time'] / 3600
    df['dt']    = [T0 + timedelta(seconds=float(s)) for s in df['Time']]
    # Добавляем относительную влажность
    df['RH'] = df['vpAir'] / (610.78 * np.exp(17.27 * df['tAir'] / (df['tAir'] + 237.3))) * 100
    df['RH'] = df['RH'].clip(0, 100)
    if 'co2Air' in df.columns:
        df['co2_ppm'] = df['co2Air'] / 1.96
    return df

print('✅ Готово. Запускаем симуляции...')

In [ ]:
%%time
# ══ СИМУЛЯЦИЯ 1: НОРМАЛЬНЫЙ РЕЖИМ (cLeakage = 1e-4) ══════════════════════════
print('🔵 Запуск: нормальный режим...')
df_normal = run_sim(MAIN_JSON, DATA_CSV, t_days=1)
print(f'  tAir: {df_normal["tAir"].min():.1f} … {df_normal["tAir"].max():.1f} °C')

# ══ ПОДГОТОВКА МОДЕЛИ «ОТКРЫТАЯ ДВЕРЬ» ═══════════════════════════════════════
print('\n🔴 Создаём модель с открытой дверью...')

# Копируем модель во временную директорию и меняем cLeakage
if os.path.exists(MOD_DIR):
    shutil.rmtree(MOD_DIR)
shutil.copytree(PKG, MOD_DIR)

# Меняем cLeakage: 1e-4 → 0.05 (в 500 раз больше = открытая дверь)
with open(EXT_FILE) as f:
    content = f.read()
content_mod = content.replace('"definition" : "1e-4"', '"definition" : "0.05"', 1)
assert content_mod != content, 'Замена не удалась!'
with open(EXT_FILE, 'w') as f:
    f.write(content_mod)

print('\n🔴 Запуск: открытая дверь...')
df_door = run_sim(MOD_MAIN, DATA_CSV, t_days=1)
print(f'  tAir: {df_door["tAir"].min():.1f} … {df_door["tAir"].max():.1f} °C')

print('\n✅ Обе симуляции завершены!')

## 2. 🌡 Главный результат: что происходит с климатом теплицы

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle('🚪 Открытая дверь vs. Нормальный режим — Октябрь (Bleiswijk, NL)',
             fontsize=15, fontweight='bold', y=0.995)

# Зона «открытой двери» — весь день для наглядности
for ax in axes:
    ax.axvspan(df_normal['dt'].iloc[0], df_normal['dt'].iloc[-1],
               alpha=0.04, color='#ff7b72', label='_nolegend_')

# ── 1. Температура ────────────────────────────────────────────────────────────
ax = axes[0]
ax.fill_between(df_normal['dt'], df_normal['tAir'],
                df_door['tAir'], alpha=0.2, color='#ff7b72',
                label='Потеря тепла из-за двери')
ax.plot(df_normal['dt'], df_normal['tAir'], color='#58a6ff', lw=2.2,
        label='🔵 Норма: tAir', zorder=5)
ax.plot(df_door['dt'],   df_door['tAir'],   color='#ff7b72', lw=2.2,
        label='🔴 Открытая дверь: tAir', zorder=5)
ax.plot(df_normal['dt'], df_normal['tOut'], color='#8b949e', lw=1.2,
        ls='--', alpha=0.8, label='⬛ Улица: tOut')
ax.set_ylabel('Температура, °C')
ax.set_title('Температура воздуха в теплице')
ax.legend(loc='upper right', ncol=2, fontsize=9)
ax.grid(True, alpha=0.35)
ax.axhline(12, color='#f0883e', ls=':', lw=1.5, alpha=0.7)
ax.text(df_normal['dt'].iloc[-1], 12.5, '⚠️ Критично для томатов: <12°C',
        ha='right', fontsize=8, color='#f0883e')

# ── 2. Влажность ──────────────────────────────────────────────────────────────
ax = axes[1]
ax.fill_between(df_normal['dt'], df_normal['RH'], df_door['RH'],
                alpha=0.2, color='#79c0ff')
ax.plot(df_normal['dt'], df_normal['RH'], color='#58a6ff', lw=2.2,
        label='🔵 Норма: RH')
ax.plot(df_door['dt'],   df_door['RH'],   color='#ff7b72', lw=2.2,
        label='🔴 Открытая дверь: RH')
ax.axhline(90, color='#ff7b72', ls='--', lw=1.5, alpha=0.8)
ax.axhline(85, color='#f0883e', ls=':', lw=1.5, alpha=0.6)
ax.text(df_normal['dt'].iloc[-1], 90.5, '⚠️ Риск конденсации: >90%',
        ha='right', fontsize=8, color='#ff7b72')
ax.set_ylabel('Влажность, %')
ax.set_ylim(50, 105)
ax.set_title('Относительная влажность воздуха')
ax.legend(loc='upper right', ncol=2, fontsize=9)
ax.grid(True, alpha=0.35)

# ── 3. CO₂ ────────────────────────────────────────────────────────────────────
ax = axes[2]
if 'co2_ppm' in df_normal.columns and 'co2_ppm' in df_door.columns:
    ax.fill_between(df_normal['dt'], df_normal['co2_ppm'], df_door['co2_ppm'],
                    alpha=0.25, color='#3fb950')
    ax.plot(df_normal['dt'], df_normal['co2_ppm'], color='#58a6ff', lw=2.2,
            label='🔵 Норма: CO₂')
    ax.plot(df_door['dt'],   df_door['co2_ppm'],   color='#ff7b72', lw=2.2,
            label='🔴 Открытая дверь: CO₂')
    ax.axhline(400, color='#8b949e', ls='--', lw=1.2, alpha=0.7,
               label='Атм. уровень CO₂ (~400 ppm)')
    ax.set_ylabel('CO₂, ppm')
    ax.set_title('Концентрация CO₂ (чем больше — тем активнее фотосинтез)')
    ax.legend(loc='upper right', ncol=2, fontsize=9)
    ax.grid(True, alpha=0.35)
else:
    ax.plot(df_normal['dt'], df_normal['vpAir'], color='#58a6ff', lw=2, label='🔵 Норма: vpAir')
    ax.plot(df_door['dt'],   df_door['vpAir'],   color='#ff7b72', lw=2, label='🔴 Открытая дверь: vpAir')
    ax.set_ylabel('Давление пара, Па')
    ax.set_title('Давление пара (влажность воздуха)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.35)

# ── 4. Температура труб отопления ─────────────────────────────────────────────
ax = axes[3]
ax.plot(df_normal['dt'], df_normal['tPipe'], color='#58a6ff', lw=2.2,
        label='🔵 Норма: tPipe')
ax.plot(df_door['dt'],   df_door['tPipe'],   color='#ff7b72', lw=2.2,
        label='🔴 Открытая дверь: tPipe')
ax.fill_between(df_normal['dt'], df_normal['tPipe'], df_door['tPipe'],
                alpha=0.2, color='#f0883e', label='Перегрузка котла')
ax.set_ylabel('Температура, °C')
ax.set_title('Трубы отопления — котёл пытается компенсировать потери')
ax.legend(loc='upper right', ncol=2, fontsize=9)
ax.grid(True, alpha=0.35)

# Форматирование оси X
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[-1].xaxis.set_major_locator(mdates.HourLocator(interval=2))
axes[-1].set_xlabel('Время суток (20 октября 2009)')

plt.tight_layout()
plt.savefig('open_door_climate.png', dpi=130, bbox_inches='tight')
plt.show()

## 3. 🍅 Влияние на урожай томатов

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('🍅 Последствия для урожая: нормальный режим vs. открытая дверь',
             fontsize=13, y=0.98)

# 1. Буфер ассимилятов (скорость фотосинтеза → накопление)
ax = axes[0, 0]
ax.plot(df_normal['dt'], df_normal['cBuf']/1000, color='#58a6ff', lw=2,
        label='🔵 Норма')
ax.plot(df_door['dt'],   df_door['cBuf']/1000,   color='#ff7b72', lw=2,
        label='🔴 Открытая дверь')
ax.fill_between(df_normal['dt'],
                df_normal['cBuf']/1000, df_door['cBuf']/1000,
                alpha=0.2, color='#ff7b72', label='Потеря ассимилятов')
ax.set_ylabel('г·CH₂O / м²')
ax.set_title('Буфер ассимилятов cBuf\n(резервуар фотосинтеза)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.35)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=4))

# 2. Плоды
ax = axes[0, 1]
ax.plot(df_normal['dt'], df_normal['cFruit']/1000, color='#58a6ff', lw=2,
        label='🔵 Норма: плоды')
ax.plot(df_door['dt'],   df_door['cFruit']/1000,   color='#ff7b72', lw=2,
        label='🔴 Открытая дверь')
ax.set_ylabel('г·CH₂O / м²')
ax.set_title('Биомасса плодов cFruit')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.35)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=4))

# 3. Температурный интеграл культуры
ax = axes[1, 0]
ax.plot(df_normal['dt'], df_normal['tCanSum'], color='#58a6ff', lw=2,
        label='🔵 Норма')
ax.plot(df_door['dt'],   df_door['tCanSum'],   color='#ff7b72', lw=2,
        label='🔴 Открытая дверь')
ax.fill_between(df_normal['dt'],
                df_normal['tCanSum'], df_door['tCanSum'],
                alpha=0.2, color='#ff7b72', label='Недобор тепловых единиц')
ax.set_ylabel('°C · сут')
ax.set_title('Температурный интеграл tCanSum\n(определяет скорость развития)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.35)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=4))

# 4. Итоговый ущерб — таблица
ax = axes[1, 1]
ax.axis('off')

loss_tAir  = df_normal['tAir'].mean() - df_door['tAir'].mean()
loss_tCan  = df_normal['tCan'].mean() - df_door['tCan'].mean()
loss_buf   = df_normal['cBuf'].iloc[-1] - df_door['cBuf'].iloc[-1]
loss_fruit = df_normal['cFruit'].iloc[-1] - df_door['cFruit'].iloc[-1]
loss_cansum= df_normal['tCanSum'].iloc[-1] - df_door['tCanSum'].iloc[-1]

table_data = [
    ['Параметр', 'Норма', 'Открытая\nдверь', 'Разница'],
    ['tAir средняя', f"{df_normal['tAir'].mean():.1f}°C",
                     f"{df_door['tAir'].mean():.1f}°C",
                     f"−{loss_tAir:.1f}°C"],
    ['tAir мин',     f"{df_normal['tAir'].min():.1f}°C",
                     f"{df_door['tAir'].min():.1f}°C",
                     f"−{df_normal['tAir'].min()-df_door['tAir'].min():.1f}°C"],
    ['RH средняя',   f"{df_normal['RH'].mean():.0f}%",
                     f"{df_door['RH'].mean():.0f}%",
                     f"{df_door['RH'].mean()-df_normal['RH'].mean():+.0f}%"],
    ['cBuf конец',   f"{df_normal['cBuf'].iloc[-1]/1000:.1f} г/м²",
                     f"{df_door['cBuf'].iloc[-1]/1000:.1f} г/м²",
                     f"−{loss_buf/1000:.1f} г/м²"],
    ['cFruit конец', f"{df_normal['cFruit'].iloc[-1]/1000:.1f} г/м²",
                     f"{df_door['cFruit'].iloc[-1]/1000:.1f} г/м²",
                     f"−{loss_fruit/1000:.2f} г/м²"],
    ['tCanSum конец',f"{df_normal['tCanSum'].iloc[-1]:.2f} °C·сут",
                     f"{df_door['tCanSum'].iloc[-1]:.2f} °C·сут",
                     f"−{loss_cansum:.2f} °C·сут"],
]

colors_table = [['#21262d']*4] + [
    ['#161b22', '#1a3a5c', '#3a1a1a', '#3a1a1a'] for _ in range(len(table_data)-1)
]
tbl = ax.table(
    cellText=table_data,
    cellLoc='center',
    loc='center',
    cellColours=colors_table,
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.5)
tbl.scale(1.15, 2.0)
# Заголовок
for j in range(4):
    tbl[(0, j)].set_text_props(fontweight='bold', color='#e6edf3')
# Остальные ячейки
for i in range(1, len(table_data)):
    for j in range(4):
        tbl[(i, j)].set_text_props(color='#e6edf3')
    # Колонка «Разница» красным
    tbl[(i, 3)].set_text_props(color='#ff7b72', fontweight='bold')

ax.set_title('📊 Итоговый ущерб за 24 часа', fontsize=11, pad=15)

plt.tight_layout()
plt.savefig('open_door_crop.png', dpi=130, bbox_inches='tight')
plt.show()

## 4. 🎛 Интерактивно: как долго дверь была открыта?

Используем результаты двух симуляций для аппроксимации промежуточных сценариев.

In [ ]:
out = widgets.Output()

w_open_h  = widgets.IntSlider(value=2, min=0, max=23, step=1,
                               description='Открыли (ч):',
                               layout=Layout(width='420px'),
                               style={'description_width': '120px'})
w_dur_h   = widgets.IntSlider(value=3, min=1, max=12, step=1,
                               description='Длительность (ч):',
                               layout=Layout(width='420px'),
                               style={'description_width': '120px'})
w_factor  = widgets.FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05,
                                 description='Степень открытия:',
                                 layout=Layout(width='420px'),
                                 style={'description_width': '120px'})
w_var     = widgets.Dropdown(
    options=[('🌡 Температура воздуха (tAir)', 'tAir'),
             ('🌿 Температура полога (tCan)', 'tCan'),
             ('💧 Относительная влажность (RH)', 'RH'),
             ('🔥 Трубы отопления (tPipe)', 'tPipe'),
             ('🔋 Буфер ассимилятов (cBuf)', 'cBuf'),
             ('🍅 Биомасса плодов (cFruit)', 'cFruit'),
             ('🌱 Темп. интеграл (tCanSum)', 'tCanSum')],
    value='tAir',
    description='Переменная:',
    layout=Layout(width='420px'),
    style={'description_width': '120px'}
)

UNITS = {
    'tAir': '°C', 'tCan': '°C', 'RH': '%', 'tPipe': '°C',
    'cBuf': 'мг/м²', 'cFruit': 'мг/м²', 'tCanSum': '°C·сут'
}
SCALE = {'cBuf': 1/1000, 'cFruit': 1/1000}

def plot_scenario(open_h, dur_h, factor, var):
    with out:
        clear_output(wait=True)

        # Время открытия и закрытия в секундах
        t_open  = open_h * 3600
        t_close = min((open_h + dur_h) * 3600, 24 * 3600)

        # Строим сценарий: интерполируем между normal и door
        # До открытия — норма; во время — mix(factor); после — норма
        t = df_normal['Time'].values
        y_n = df_normal[var].values
        y_d = df_door[var].values

        weight = np.zeros(len(t))
        mask_open = (t >= t_open) & (t <= t_close)
        weight[mask_open] = factor

        # Плавный переход ±30 мин
        from scipy.ndimage import gaussian_filter1d
        weight_smooth = gaussian_filter1d(weight, sigma=6)
        y_scenario = y_n * (1 - weight_smooth) + y_d * weight_smooth

        sc = SCALE.get(var, 1)
        unit = UNITS.get(var, '?')

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                        gridspec_kw={'height_ratios': [3, 1]})
        fig.suptitle(f'🚪 Сценарий: дверь открыта в {open_h:02d}:00, '
                     f'закрыта в {min(open_h+dur_h,24):02d}:00 '
                     f'(степень открытия {factor*100:.0f}%)', fontsize=12)

        # Зона открытой двери
        dt_open  = T0 + timedelta(seconds=int(t_open))
        dt_close = T0 + timedelta(seconds=int(t_close))
        ax1.axvspan(dt_open, dt_close, alpha=0.12, color='#ff7b72', label='Дверь открыта')
        ax2.axvspan(dt_open, dt_close, alpha=0.12, color='#ff7b72')

        # Линии
        ax1.plot(df_normal['dt'], y_n*sc, color='#58a6ff', lw=1.8,
                 ls='--', alpha=0.6, label='🔵 Норма')
        ax1.plot(df_door['dt'],   y_d*sc, color='#ff7b72', lw=1.8,
                 ls='--', alpha=0.6, label='🔴 Полностью открыта (500×)')
        ax1.plot(df_normal['dt'], y_scenario*sc, color='#3fb950', lw=2.5,
                 label=f'🟢 Ваш сценарий ({factor*100:.0f}% от max)')

        # Вертикальные отметки
        ax1.axvline(dt_open,  color='#ff7b72', ls=':', lw=2)
        ax1.axvline(dt_close, color='#3fb950', ls=':', lw=2)

        ax1.set_ylabel(f'{var} [{unit}]')
        ax1.legend(loc='upper right', fontsize=9, ncol=2)
        ax1.grid(True, alpha=0.35)

        # Отклонение от нормы
        delta = (y_scenario - y_n) * sc
        ax2.fill_between(df_normal['dt'], delta, alpha=0.5,
                         color=np.where(delta >= 0, '#3fb950', '#ff7b72')[0])
        ax2.plot(df_normal['dt'], delta, color='#ffa657', lw=1.2)
        ax2.axhline(0, color='#58a6ff', lw=1.5)
        ax2.set_ylabel(f'Δ{var} [{unit}]')
        ax2.set_title(f'Отклонение от нормы | '
                      f'Макс: {delta.min():.2f} {unit}')
        ax2.grid(True, alpha=0.35)

        for ax in [ax1, ax2]:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
        ax2.set_xlabel('Время суток')

        # Статистика в текстовом поле
        delta_max = abs(delta).max()
        delta_mean = abs(delta[mask_open]).mean() if mask_open.any() else 0
        info = (f'Макс. отклонение:  {delta_max:.2f} {unit}\n'
                f'Среднее (окно):    {delta_mean:.2f} {unit}\n'
                f'Длительность:      {dur_h} ч')
        ax1.text(0.01, 0.98, info, transform=ax1.transAxes,
                 fontsize=8.5, va='top', color='#e6edf3',
                 bbox=dict(boxstyle='round', facecolor='#161b22',
                           alpha=0.9, edgecolor='#30363d'))

        plt.tight_layout()
        plt.show()

ui = VBox([
    widgets.HTML('<h4 style="color:#e6edf3">⚙️ Параметры сценария</h4>'),
    HBox([w_open_h, w_dur_h]),
    HBox([w_factor, w_var]),
])

widgets.interactive_output(
    plot_scenario,
    {'open_h': w_open_h, 'dur_h': w_dur_h, 'factor': w_factor, 'var': w_var}
)
display(ui, out)
plot_scenario(2, 3, 0.9, 'tAir')

## 5. ⏱ Как быстро теплица реагирует?

Здесь смотрим на **скорость изменения** — когда именно критический порог будет пройден.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('⏱ Скорость реакции теплицы при открытой двери', fontsize=13)

# ── Левый: tAir и tOut схождение ──────────────────────────────────────────────
ax = axes[0]

# Считаем разницу tAir и tOut
delta_temp_normal = df_normal['tAir'] - df_normal['tOut']
delta_temp_door   = df_door['tAir']   - df_door['tOut']

ax.fill_between(df_normal['dt'], delta_temp_normal, alpha=0.3,
                color='#58a6ff', label='🔵 Норма: tAir − tOut')
ax.fill_between(df_door['dt'],   delta_temp_door,   alpha=0.3,
                color='#ff7b72', label='🔴 Открытая дверь: tAir − tOut')
ax.plot(df_normal['dt'], delta_temp_normal, color='#58a6ff', lw=2)
ax.plot(df_door['dt'],   delta_temp_door,   color='#ff7b72', lw=2)
ax.axhline(0, color='#3fb950', ls='--', lw=1.5,
           label='Нулевое превышение = полный выравнивание')
ax.axhline(5, color='#e3b341', ls=':', lw=1.2, label='Минимально допустимый ΔT = 5°C')
ax.set_ylabel('tAir − tOut, °C')
ax.set_title('Превышение над уличной температурой')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.35)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))

# ── Правый: тепловые потоки — сколько тепла теряем ───────────────────────────
ax = axes[1]

# Тепловые потери ≈ cPAir * ρAir * fVent * (tAir - tOut)
# Примерно оцениваем по разности энергии отопления (tPipe как прокси)
# Нормируем разницу tPipe × объём как прокси теплопотерь
power_normal = (df_normal['tPipe'] - df_normal['tAir']).clip(0)
power_door   = (df_door['tPipe']   - df_door['tAir']).clip(0)

ax.fill_between(df_normal['dt'], power_normal, alpha=0.3,
                color='#58a6ff', label='🔵 Норма: ΔT труб')
ax.fill_between(df_door['dt'],   power_door,   alpha=0.3,
                color='#ff7b72', label='🔴 Открытая дверь: ΔT труб')
ax.plot(df_normal['dt'], power_normal, color='#58a6ff', lw=2)
ax.plot(df_door['dt'],   power_door,   color='#ff7b72', lw=2)
ax.set_ylabel('tPipe − tAir, °C (нагрузка на котёл)')
ax.set_title('Нагрузка на систему отопления\n(чем больше — тем больше тратится газа/электричества)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.35)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))

# Аннотация перерасхода
extra_load = (power_door - power_normal).clip(0).mean()
ax.text(0.02, 0.97,
        f'Средний перерасход: +{extra_load:.1f}°C ΔT\n(~пропорционально расходу топлива)',
        transform=ax.transAxes, fontsize=9, va='top', color='#ffa657',
        bbox=dict(boxstyle='round', facecolor='#161b22', alpha=0.9, edgecolor='#ff7b72'))

plt.tight_layout()
plt.show()

# Финальная сводка
print('\n' + '='*60)
print('📋 СВОДКА ПОСЛЕДСТВИЙ ОТКРЫТОЙ ДВЕРИ')
print('='*60)
print(f'  Средняя потеря температуры: {df_normal["tAir"].mean()-df_door["tAir"].mean():.1f} °C')
print(f'  Минимальная tAir (открытая дверь): {df_door["tAir"].min():.1f} °C')
print(f'  (Критично для томатов при < 12°C)')
print(f'  Средняя влажность (норма/дверь): {df_normal["RH"].mean():.0f}% / {df_door["RH"].mean():.0f}%')
if 'co2_ppm' in df_normal.columns:
    print(f'  CO₂ (норма/дверь): {df_normal["co2_ppm"].mean():.0f}/{df_door["co2_ppm"].mean():.0f} ppm')
print(f'  Потеря биомассы плодов: {(df_normal["cFruit"].iloc[-1]-df_door["cFruit"].iloc[-1])/1000:.2f} г·CH₂O/м²')
print(f'  Потеря тепл. единиц: {df_normal["tCanSum"].iloc[-1]-df_door["tCanSum"].iloc[-1]:.3f} °C·сут')
print('='*60)

## 📝 Выводы

| Показатель | Последствие | Критический порог |
|---|---|---|
| **tAir** | Падает к уличной (3–11°C) | < 12°C — стресс у томатов |
| **Влажность** | Растёт → риск конденсации | > 90% → грибковые болезни |
| **CO₂** | Падает до атмосферного (400 ppm) | < 600 ppm → замедление фотосинтеза |
| **Отопление** | Котёл перегружается | Перерасход энергии |
| **Биомасса** | Недобор ассимилятов | Задержка плодоношения |

### Как использовать эти знания?
1. **Датчик температуры** у двери → тревога при резком падении tAir
2. **Цифровой двойник** сравнивает модель с датчиками → обнаруживает аномалию за 5-15 мин
3. **Система алертов**: отклонение tAir > 3°C от модели → SMS/уведомление
4. **Экономика**: 1 час открытой двери в октябре ≈ потеря X кВт·ч тепла